In this notebook, I want to implement a Lie Group Variational Integrator for an object in orbit around a planet. I will assume that the mass of the object is significantly smaller than the mass of the planet. 
I will also assume initially that the object has no external forces on it other than the force from the gravitational field caused by the planet. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la
import pandas as pd

From the LGVI for Full Body Problem paper, we can find the discrete equations of motion by expressing the discrete Lagrangian for the system, and setting the variation of the action to 0. 
$\newline$
The form is very similar to the one on SO(3), but just with additional equations that express the change in position and velocity

\begin{equation}
\begin{aligned}
x_{k+1} &= x_k + \frac{h}{m} \gamma_k - \frac{h^2}{2m} U'_k \\
\gamma_{k+1} &= \gamma_k + \frac{h}{2} U'_k + \frac{h}{2} U'_{k+1} \\ 
h S(\Pi_k + \frac{h}{2} M_k) &= F_k J_d - J_d F_k ^ T \\
\Pi_{k+1} &= F_k ^ T \Pi_k + \frac{h}{2} F_k ^ T M_k + \frac{h}{2} M_{k+1} \\
R_{k+1} &= R_k F_k
\end{aligned}
\end{equation}

We write similar functions as before

In [ ]:
def S(x): 
    # S mapping
    # skew symmetric mapping with S(x)y = cross(x,y)
    return np.array([[0., -x[2], x[1]],
                     [x[2], 0., -x[0]],
                     [-x[1], x[0], 0.]])

def c1_c2_and_derivs(a):
    if a < 1e-8:
        # use Taylor expansions for small values. 
        c1 = 1 - a*a/6 + a**4/120
        c2 = 0.5 - a*a/24 + a**4/720
        dc1 = -a/3 + a**3/30
        dc2 = -a/12 + (a**3)/120
    else:
        c1 = np.sin(a)/a
        c2 = (1 - np.cos(a)) / (a*a)
        dc1 = (a*np.cos(a) - np.sin(a)) / (a*a)
        dc2 = (a*np.sin(a) - 2*(1 - np.cos(a))) / (a**3)
    return c1, c2, dc1, dc2    

def solving_for_f(J, Pi_k, M_k, h, tol = 1e-10, maxit = 50): 
    'Newton Iteration implementation to solve for f'
    
    b = h * Pi_k + ((h**2)/2) * M_k #constant vector
    f = np.linalg.solve(J,b) # setting initial conditions

    # iterating
    for i in range(maxit):
        a = np.linalg.norm(f)
        c1, c2, dc1, dc2 = c1_c2_and_derivs(a)
        Jf = J @ f
        cross = np.cross(f, Jf)

        # G(f)
        G = c1 * Jf + c2 * cross - b

        if np.linalg.norm(G) < tol:
            return f, True, i

        # Jacobian DG(f)
        f_transpose = np.transpose(f)
        S_terms = -S(Jf) + S(f) @ J
        DG = c1 * J + c2 * S_terms # terms 1 and 3
        # terms 2 and 4 if a is non zero
        if a > 0:
            DG += (dc1 * (Jf @ f_transpose) / a) + dc2 * (cross @ f_transpose) / a
        else: 
            pass

        delta = np.linalg.solve(DG, G)
        f = f - delta #new f

        if np.linalg.norm(delta) < 1e-12:
            return f, True, i+1
    
    return f, False, maxit


def F_from_f(f):
    a = np.linalg.norm(f)
    c1, c2, dc1, dc2 = c1_c2_and_derivs(a)
    Sf = S(f)
    F = np.eye(3) + c1 * Sf + c2 * (Sf @ Sf)
    return F